# NB-502: DPO Alignment

Direct Preference Optimization for LLM alignment.

## Learning Objectives
- Preference data format
- DPO algorithm
- Training with DPO
- Comparing with RLHF

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

print("Libraries imported")

## 1. Create Preference Dataset

In [ ]:
# TODO: Sample preference pairs
preference_data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris.",
        "rejected": "France is in Europe.",
    },
    {
        "prompt": "Explain machine learning.",
        "chosen": "Machine learning is a subset of AI that enables systems to learn from data.",
        "rejected": "I don't know much about that topic.",
    },
]

dataset = Dataset.from_list(preference_data)
print(f"Preference pairs: {len(dataset)}")

## 2. DPO Loss Function

In [ ]:
def dpo_loss(policy_logratios, ref_logratios, beta=0.1):
    """
    Compute DPO loss.

    DPO Loss = -log σ(β * (log π(y_chosen|x) - log π(y_rejected|x)))
              -log σ(β * (log π_ref(y_rejected|x) - log π_ref(y_chosen|x)))
    """
    # TODO: Compute DPO loss
    chosen_logratios = policy_logratios[0] - ref_logratios[0]
    rejected_logratios = policy_logratios[1] - ref_logratios[1]

    losses = -torch.log(torch.sigmoid(beta * (chosen_logratios - rejected_logratios)))

    return losses.mean()

# Test
policy_lr = torch.tensor([[1.0], [0.5]])
ref_lr = torch.tensor([[0.8], [0.6]])

loss = dpo_loss(policy_lr, ref_lr)
print(f"DPO loss: {loss.item():.4f}")

## 3. DPO Training with TRL

In [ ]:
# TODO: Use TRL library for DPO
from trl import DPOTrainer, DPOConfig

# Configuration
dpo_config = DPOConfig(
    learning_rate=1e-5,
    beta=0.1,
    max_length=512,
)

print(f"DPO config: beta={dpo_config.beta}, lr={dpo_config.learning_rate}")

## 4. Evaluate Aligned Model

In [ ]:
def evaluate_alignment(model, tokenizer, test_prompts):
    """Evaluate model alignment."""

    results = []

    for prompt in test_prompts:
        inputs = tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=50)

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results.append({"prompt": prompt, "response": response})

    return results

print("Evaluation function defined")

## Exercise

1. Create larger preference dataset
2. Compare DPO vs PPO
3. Implement ORPO (Odds Ratio Preference Optimization)